# Focolai attivi in Puglia · NASA FIRMS → CSV

Scarica i **punti di calore** rilevati dai satelliti NASA (VIIRS e MODIS, tempo quasi reale) **solo per la Puglia** e li salva in CSV.

**Cosa produce:** un solo file, `firms_puglia.csv`, con un punto di calore per riga: ora UTC e italiana, comune, provincia,
satellite, affidabilità, potenza, area industriale e numero del possibile focolaio.
È lo stesso formato che l'automazione GitHub del sito aggiorna ogni 30 minuti in `data/firms_puglia.csv`:
per il server MCP conviene leggere quello, che è sempre aggiornato; questo notebook serve per scaricarlo a mano o per date passate.

**Prima volta: la chiave**
1. Richiedi la chiave gratuita (MAP_KEY) su https://firms.modaps.eosdis.nasa.gov/api/area/ (pulsante *Get MAP_KEY*).
2. In Colab clicca l'icona della **chiave** 🔑 nella barra a sinistra → *Aggiungi nuovo secret*: nome `FIRMS_MAP_KEY`, valore la chiave, e attiva *Accesso al notebook*.

Non scrivere mai la chiave dentro le celle: se condividi il notebook, la vedrebbero tutti.

Poi: *Runtime → Esegui tutte*.

## 1. Impostazioni

### Nota sui giorni UTC

L'API di NASA FIRMS ragiona in **giorni UTC** (il tempo universale), non in "ultime N ore".
`GIORNI = 1` significa "solo la data di oggi in UTC", dalla mezzanotte UTC a ora.

In Italia la mezzanotte UTC cade alle **2:00 con l'ora legale** e all'**1:00 con l'ora solare**.
Quindi, per esempio, eseguendo il notebook alle 3 di notte con `GIORNI = 1` si ottiene solo
l'ultima ora circa di dati.

- **Per le ultime 24 ore:** usa `GIORNI = 2` e filtra la colonna `ore_fa < 24`.
- **Per le ultime 48 ore:** usa `GIORNI = 3` e filtra `ore_fa < 48`.

Anche gli orari grezzi della NASA (`acq_date`, `acq_time`) sono in UTC; la colonna
`data_ora_italia` li riporta già all'ora italiana.

In [ ]:
GIORNI = 2                 # @param {type:"slider", min:1, max:5, step:1}
# Giorni UTC da scaricare: 1 = solo oggi (UTC), 2 = oggi e ieri, ... fino a 5 (limite dell'API).

DATA_INIZIO = ""           # @param {type:"string"}
# Vuoto = dati piu' recenti. Oppure una data AAAA-MM-GG: restituisce da quella data per GIORNI giorni.

FONTI = ["VIIRS_SNPP_NRT", "VIIRS_NOAA20_NRT", "VIIRS_NOAA21_NRT", "MODIS_NRT"]
# Per date vecchie (oltre qualche mese) i dati NRT non ci sono piu': usa VIIRS_SNPP_SP e MODIS_SP.

SOLO_PUGLIA = True         # @param {type:"boolean"}
# True = tiene solo i punti nei comuni pugliesi (entro TOLLERANZA_KM dalla costa/confine).

TOLLERANZA_KM = 2.0        # @param {type:"number"}
RAGGIO_FOCOLAIO_KM = 1.0   # @param {type:"number"}
SEPARATORE_CSV = ","       # @param [",", ";"]
# ";" si apre meglio con Excel in italiano.

CARTELLA_DRIVE = ""        # @param {type:"string"}
# Vuoto = scarica i file sul computer. Oppure una cartella di Google Drive, es. "incendi_puglia/firms".

# Riquadro geografico richiesto alla NASA: ovest, sud, est, nord (solo la Puglia, non il mondo)
BBOX = (14.9, 39.75, 18.55, 42.25)

# Impianti che producono calore tutto l'anno: i punti qui dentro probabilmente NON sono incendi.
# Posizioni e raggi approssimativi: verificali e aggiungine altri se necessario.
AREE_INDUSTRIALI = [
    {"nome": "area siderurgica e raffineria di Taranto", "lat": 40.505, "lon": 17.195, "r_km": 4.0},
    {"nome": "centrale termoelettrica di Brindisi Cerano", "lat": 40.563, "lon": 18.032, "r_km": 1.5},
]

## 2. Librerie

In [ ]:
import io, os, re, time, math, json
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import requests

try:
    import geopandas as gpd
except ImportError:
    !pip install -q geopandas
    import geopandas as gpd
from sklearn.cluster import DBSCAN

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
ORA = datetime.now(timezone.utc)
TIMBRO = pd.Timestamp(ORA).tz_convert("Europe/Rome").strftime("%Y%m%d_%H%M")
print("Esecuzione:", pd.Timestamp(ORA).tz_convert("Europe/Rome").strftime("%d/%m/%Y %H:%M"), "(ora italiana)")

Esecuzione: 25/09/2026 18:10 (ora italiana)


## 3. Chiave FIRMS
Letta dai Secrets di Colab. Se non la trova, la chiede in un campo nascosto (non resta salvata nel notebook).

In [ ]:
def leggi_chiave():
    try:
        from google.colab import userdata
        k = userdata.get("FIRMS_MAP_KEY")
        if k:
            return k.strip()
    except Exception as e:
        print("Secret non disponibile:", type(e).__name__, "- controlla nome e 'Accesso al notebook'.")
    if os.environ.get("FIRMS_MAP_KEY"):
        return os.environ["FIRMS_MAP_KEY"].strip()
    from getpass import getpass
    return getpass("Incolla la chiave FIRMS (MAP_KEY): ").strip()

MAP_KEY = leggi_chiave()
assert MAP_KEY, "Chiave mancante"
print("Chiave caricata:", MAP_KEY[:4] + "…" + MAP_KEY[-2:])

Chiave caricata: 4ac8…83


In [ ]:
# Facoltativo: quante richieste restano alla chiave (limite: 5000 ogni 10 minuti).
try:
    r = requests.get("https://firms.modaps.eosdis.nasa.gov/mapserver/mapkey_status/", params={"MAP_KEY": MAP_KEY}, timeout=30)
    print(r.json() if r.headers.get("content-type", "").startswith("application/json") else r.text[:200])
except Exception as e:
    print("Controllo dello stato non riuscito (non blocca il resto):", type(e).__name__)

{'transaction_limit': 5000, 'current_transactions': 16, 'transaction_interval': '10 minutes'}


## 4. Scaricamento
Una richiesta per satellite, solo sul riquadro della Puglia. Se un satellite non risponde, gli altri vengono comunque scaricati.

In [ ]:
API = "https://firms.modaps.eosdis.nasa.gov/api/area/csv"

def url_firms(fonte):
    parti = [API, MAP_KEY, fonte, ",".join(str(v) for v in BBOX), str(GIORNI)]
    if DATA_INIZIO.strip():
        parti.append(DATA_INIZIO.strip())
    return "/".join(parti)

def scarica(fonte, tentativi=3):
    ultimo = None
    for t in range(tentativi):
        try:
            r = requests.get(url_firms(fonte), timeout=(15, 120))
            if r.status_code == 200:
                testo = r.text
                if not testo.lstrip().lower().startswith("latitude"):
                    raise ValueError("risposta inattesa: " + testo.strip()[:150])
                return pd.read_csv(io.StringIO(testo), dtype={"acq_time": str})
            if r.status_code == 429 or r.status_code >= 500:
                ultimo = f"HTTP {r.status_code}"
            else:
                raise ValueError(f"HTTP {r.status_code}: {r.text.strip()[:150]}")
        except requests.exceptions.RequestException as e:
            ultimo = type(e).__name__
        time.sleep(10 * 2 ** t)
    raise RuntimeError(f"{tentativi} tentativi falliti ({ultimo})")

tabelle, stato = [], []
for f in FONTI:
    try:
        df = scarica(f)
        df["fonte"] = f
        tabelle.append(df)
        stato.append({"fonte": f, "esito": "ok", "punti_nel_riquadro": len(df), "errore": ""})
    except Exception as e:
        stato.append({"fonte": f, "esito": "errore", "punti_nel_riquadro": 0, "errore": str(e).replace(MAP_KEY, "***")})
stato = pd.DataFrame(stato)
display(stato)
assert (stato.esito == "ok").any(), "Nessun satellite ha risposto: controlla la chiave e riprova tra qualche minuto."
grezzi = pd.concat(tabelle, ignore_index=True) if tabelle else pd.DataFrame()
print("Punti nel riquadro:", len(grezzi))

,fonte,esito,punti_nel_riquadro,errore
0,VIIRS_SNPP_NRT,ok,73,
1,VIIRS_NOAA20_NRT,ok,76,
2,VIIRS_NOAA21_NRT,ok,58,
3,MODIS_NRT,ok,6,


Punti nel riquadro: 213


## 5. Pulizia e colonne leggibili

In [ ]:
NOMI_FONTE = {"VIIRS_SNPP_NRT": "VIIRS S-NPP", "VIIRS_NOAA20_NRT": "VIIRS NOAA-20", "VIIRS_NOAA21_NRT": "VIIRS NOAA-21",
              "MODIS_NRT": "MODIS", "VIIRS_SNPP_SP": "VIIRS S-NPP (archivio)", "MODIS_SP": "MODIS (archivio)"}

def affidabilita(v):
    s = str(v).strip().lower()
    if s in ("h", "high"): return "alta"
    if s in ("n", "nominal"): return "nominale"
    if s in ("l", "low"): return "bassa"
    try:
        n = float(s)                       # MODIS: percentuale 0-100
    except ValueError:
        return "n.d."
    return "alta" if n >= 80 else ("nominale" if n >= 30 else "bassa")

g = grezzi.copy()
ora_str = g["acq_time"].astype(str).str.strip().str.zfill(4)
g["data_ora_utc"] = pd.to_datetime(g["acq_date"] + " " + ora_str.str[:2] + ":" + ora_str.str[2:], utc=True)
g["data_ora_italia"] = g["data_ora_utc"].dt.tz_convert("Europe/Rome")
g["ore_fa"] = ((pd.Timestamp(ORA) - g["data_ora_utc"]).dt.total_seconds() / 3600).round(1)
g["satellite"] = g["fonte"].map(NOMI_FONTE).fillna(g["fonte"])
g["affidabilita"] = g["confidence"].map(affidabilita)
g["temperatura_brillanza_k"] = g["bright_ti4"] if "bright_ti4" in g else np.nan
if "brightness" in g:
    g["temperatura_brillanza_k"] = g["temperatura_brillanza_k"].fillna(g["brightness"])
g["passaggio"] = g["daynight"].map({"D": "diurno", "N": "notturno"})
g = g.drop_duplicates(subset=["latitude", "longitude", "data_ora_utc", "fonte"])
print(len(g), "punti dopo la rimozione dei duplicati")

213 punti dopo la rimozione dei duplicati


## 6. Solo la Puglia: comune e provincia
Confini comunali ISTAT (openpolis). I pixel appena fuori dalla costa vengono assegnati al comune più vicino entro `TOLLERANZA_KM`.

In [ ]:
URL_COMUNI = "https://raw.githubusercontent.com/openpolis/geojson-italy/master/geojson/limits_R_16_municipalities.geojson"
comuni = gpd.read_file(URL_COMUNI)[["name", "prov_acr", "geometry"]].rename(columns={"name": "comune", "prov_acr": "provincia"})

punti = gpd.GeoDataFrame(g, geometry=gpd.points_from_xy(g["longitude"], g["latitude"]), crs=4326)
if len(punti):
    dentro = gpd.sjoin(punti, comuni, how="left", predicate="within").drop(columns="index_right")
    dentro = dentro[~dentro.index.duplicated()]
    dentro["distanza_confine_km"] = np.where(dentro["comune"].notna(), 0.0, np.nan)
    fuori = dentro["comune"].isna()
    if fuori.any():
        vicini = gpd.sjoin_nearest(punti.loc[fuori].to_crs(32633), comuni.to_crs(32633), how="left",
                                   max_distance=TOLLERANZA_KM * 1000, distance_col="d_m")
        vicini = vicini[~vicini.index.duplicated()]
        dentro.loc[vicini.index, ["comune", "provincia"]] = vicini[["comune", "provincia"]].values
        dentro.loc[vicini.index, "distanza_confine_km"] = (vicini["d_m"] / 1000).round(2).values
else:
    dentro = punti.assign(comune=None, provincia=None, distanza_confine_km=np.nan)

in_puglia = dentro["comune"].notna()
print(f"In Puglia: {in_puglia.sum()} | fuori (Basilicata, Molise, Campania, mare...): {(~in_puglia).sum()}")
puglia = dentro[in_puglia].copy() if SOLO_PUGLIA else dentro.copy()

In Puglia: 157 | fuori (Basilicata, Molise, Campania, mare...): 56


## 7. Aree industriali e possibili focolai

In [ ]:
def dist_km(lat1, lon1, lat2, lon2):
    kx = 111.32 * np.cos(np.radians((lat1 + lat2) / 2))
    return np.hypot((lon2 - lon1) * kx, (lat2 - lat1) * 110.57)

puglia["area_industriale"] = ""
for a in AREE_INDUSTRIALI:
    vicino = dist_km(puglia["latitude"], puglia["longitude"], a["lat"], a["lon"]) <= a["r_km"]
    puglia.loc[vicino, "area_industriale"] = a["nome"]
if "type" in puglia:                             # solo nei dati d'archivio: 2 = fonte di calore statica
    puglia.loc[puglia["type"].astype(str) == "2", "area_industriale"] = "fonte statica secondo NASA"

# Possibili focolai: punti a meno di RAGGIO_FOCOLAIO_KM (esclusi quelli industriali)
puglia["focolaio_id"] = -1
non_ind = puglia["area_industriale"] == ""
if non_ind.sum():
    X = np.radians(puglia.loc[non_ind, ["latitude", "longitude"]].values)
    etichette = DBSCAN(eps=RAGGIO_FOCOLAIO_KM / 6371.0, min_samples=1, metric="haversine").fit_predict(X)
    puglia.loc[non_ind, "focolaio_id"] = etichette + 1
print(puglia["area_industriale"].replace("", "nessuna").value_counts())

area_industriale
nessuna                                     146
area siderurgica e raffineria di Taranto     11
Name: count, dtype: int64


## 8. Tabella finale
Colonne pensate per essere interrogate (per esempio da un server MCP):
- `data_ora_utc` in formato ISO (`2026-09-25T12:32:00Z`): serve per calcolare **al momento della domanda** da quante ore è stato visto il punto;
- `area_industriale` vuota = non è un impianto noto;
- `focolaio_id`: punti con lo stesso numero formano un possibile focolaio (0 = area industriale). I numeri cambiano a ogni scaricamento;
- `scaricato_il_utc`: quando è stato generato il file.

In [ ]:
COLONNE = {"data_ora_utc": "data_ora_utc", "data_ora_italia": "data_ora_italia", "comune": "comune", "provincia": "provincia",
           "latitude": "lat", "longitude": "lon", "satellite": "satellite", "affidabilita": "affidabilita",
           "frp": "frp_mw", "passaggio": "passaggio", "area_industriale": "area_industriale", "focolaio_id": "focolaio_id"}
tab = puglia[list(COLONNE)].rename(columns=COLONNE).sort_values("data_ora_utc", ascending=False).reset_index(drop=True)
tab["focolaio_id"] = tab["focolaio_id"].clip(lower=0).astype(int)
nz = tab["focolaio_id"] > 0                     # numeri in ordine di rilevamento piu' recente (1 = il piu' recente)
ordine = {v: i + 1 for i, v in enumerate(pd.unique(tab.loc[nz, "focolaio_id"]))}
tab.loc[nz, "focolaio_id"] = tab.loc[nz, "focolaio_id"].map(ordine)
tab["data_ora_utc"] = tab["data_ora_utc"].dt.strftime("%Y-%m-%dT%H:%M:00Z")
tab["data_ora_italia"] = tab["data_ora_italia"].dt.strftime("%Y-%m-%d %H:%M")
tab[["lat", "lon"]] = tab[["lat", "lon"]].round(5)
tab["scaricato_il_utc"] = pd.Timestamp(ORA).strftime("%Y-%m-%dT%H:%M:%SZ")

ore = (pd.Timestamp(ORA) - pd.to_datetime(tab["data_ora_utc"], utc=True)).dt.total_seconds() / 3600
attivi = tab[(ore < 24) & (tab.area_industriale == "") & (tab.affidabilita != "bassa")]
print(f"{len(tab)} punti di calore in Puglia; nelle ultime 24 ore {len(attivi)} punti in "
      f"{attivi.focolaio_id.nunique()} possibili focolai (esclusi impianti industriali e bassa affidabilità)")
display(tab.head(10))

157 punti di calore in Puglia; nelle ultime 24 ore 19 punti in 9 possibili focolai (esclusi impianti industriali e bassa affidabilità)


,data_ora_utc,data_ora_italia,comune,provincia,lat,lon,satellite,affidabilita,frp_mw,passaggio,area_industriale,focolaio_id,scaricato_il_utc
0,2026-09-25T12:32:00Z,2026-09-25 14:32,Ascoli Satriano,FG,41.09771,15.72876,VIIRS NOAA-20,nominale,12.81,diurno,,1,2026-09-25T16:10:03Z
1,2026-09-25T12:32:00Z,2026-09-25 14:32,Manfredonia,FG,41.54874,15.76771,VIIRS NOAA-20,nominale,3.00,diurno,,2,2026-09-25T16:10:03Z
2,2026-09-25T11:36:00Z,2026-09-25 13:36,Troia,FG,41.37293,15.28656,VIIRS NOAA-21,nominale,4.02,diurno,,3,2026-09-25T16:10:03Z
3,2026-09-25T11:36:00Z,2026-09-25 13:36,Poggiorsini,BA,40.91978,16.23552,VIIRS NOAA-21,nominale,4.24,diurno,,4,2026-09-25T16:10:03Z
4,2026-09-25T10:51:00Z,2026-09-25 12:51,Poggiorsini,BA,40.92120,16.23360,VIIRS NOAA-20,nominale,5.23,diurno,,4,2026-09-25T16:10:03Z
5,2026-09-25T01:07:00Z,2026-09-25 03:07,Lucera,FG,41.43704,15.29604,VIIRS NOAA-20,nominale,0.49,notturno,,5,2026-09-25T16:10:03Z
6,2026-09-25T01:07:00Z,2026-09-25 03:07,Serracapriola,FG,41.75963,15.17959,VIIRS NOAA-20,nominale,0.42,notturno,,6,2026-09-25T16:10:03Z
7,2026-09-25T01:07:00Z,2026-09-25 03:07,Chieuti,FG,41.83707,15.09484,VIIRS NOAA-20,nominale,0.85,notturno,,7,2026-09-25T16:10:03Z
8,2026-09-25T01:07:00Z,2026-09-25 03:07,Chieuti,FG,41.88332,15.20028,VIIRS NOAA-20,nominale,5.07,notturno,,8,2026-09-25T16:10:03Z
9,2026-09-25T01:07:00Z,2026-09-25 03:07,Apricena,FG,41.78530,15.35172,VIIRS NOAA-20,nominale,1.86,notturno,,9,2026-09-25T16:10:03Z


## 9. Salvataggio CSV

In [ ]:
NOME = "firms_puglia.csv"
dec = "," if SEPARATORE_CSV == ";" else "."
if CARTELLA_DRIVE.strip():
    from google.colab import drive
    drive.mount("/content/drive")
    dest = os.path.join("/content/drive/MyDrive", CARTELLA_DRIVE.strip())
else:
    dest = "/content" if os.path.isdir("/content") else "."
os.makedirs(dest, exist_ok=True)
percorso = os.path.join(dest, NOME)
tab.to_csv(percorso, index=False, sep=SEPARATORE_CSV, decimal=dec, encoding="utf-8")
print("Salvato", percorso, tab.shape)
if not CARTELLA_DRIVE.strip():
    try:
        from google.colab import files
        files.download(percorso)
    except ImportError:
        pass

Salvato /content/firms_puglia.csv (157, 13)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>